# Module 02: Network Protocols Transport API Paradigms — Interactive Laboratory

Every cell below runs the module's **real** implementation from
`project_solution/rpc_protocol_engine.py`. Nothing here prints a claim it has not verified.

What you will do:

1. Load the engine and inspect what it actually exports.
2. Run its primary workflow and check the assertions that define correctness.
3. **Commit to a prediction**, then run the cell that tests it.
4. Measure a property rather than asserting one.
5. Fix a deliberately broken cell in place.

> The code in cells 4, 6 and 8 is lifted from this module's own test suite, so it
> cannot drift from the implementation. If the API changes, those tests fail
> first and this notebook is regenerated from them.


## 1. Load the engine and introspect it

Rather than trusting a hardcoded list of class names, ask the module what it
actually contains.


In [ ]:
import inspect
import sys
from pathlib import Path

sys.path.insert(0, str(Path('.').resolve() / 'project_solution'))
import rpc_protocol_engine

classes = [n for n, o in inspect.getmembers(rpc_protocol_engine, inspect.isclass)
           if o.__module__ == 'rpc_protocol_engine']
functions = [n for n, o in inspect.getmembers(rpc_protocol_engine, inspect.isfunction)
             if o.__module__ == 'rpc_protocol_engine']

print('module   : rpc_protocol_engine')
print(f'classes  : {classes}')
print(f'functions: {functions}')
print()
for name in classes:
    obj = getattr(rpc_protocol_engine, name)
    try:
        sig = inspect.signature(obj.__init__)
        params = [p for p in sig.parameters if p != 'self']
    except (TypeError, ValueError):
        params = ['<builtin>']
    print(f'  {name}({", ".join(params)})')

## 2. Baseline: Frame encoding and header decoding

This is the module's own `test_frame_encoding_and_header_decoding` — real instantiation, real calls, real
assertions. If it runs clean, the property it encodes holds.


In [ ]:
import asyncio

import pytest
from rpc_protocol_engine import (
    HEADER_SIZE,
    FrameType,
    RPCClient,
    RPCFrame,
    RPCFrameCodec,
    RPCServer,
)


def frame_codec() -> type[RPCFrameCodec]:
    return RPCFrameCodec

_make_frame_codec = frame_codec

frame_codec = _make_frame_codec()

correlation_id = b"0123456789abcdef"
payload = b"Hello Distributed Systems!"
frame = RPCFrame(
    frame_type=FrameType.REQUEST,
    correlation_id=correlation_id,
    payload=payload,
)

encoded = frame_codec.encode(frame)
assert len(encoded) == HEADER_SIZE + len(payload)

frame_type, decoded_corr_id, payload_len = frame_codec.decode_header(
    encoded[:HEADER_SIZE]
)
assert frame_type == FrameType.REQUEST
assert decoded_corr_id == correlation_id
assert payload_len == len(payload)

print('PASSED: test_frame_encoding_and_header_decoding')

## 3. 🔮 Prediction — commit before you run

Predict which is larger: the byte overhead of 1,000 JSON-over-HTTP/1.1 requests, or 1,000 gRPC-over-HTTP/2 calls carrying the same fields. By roughly what factor?

Write your answer down. An uncommitted guess teaches nothing, because you will
retro-fit it to whatever the next cell prints.

The next cell runs `test_invalid_magic_byte_raises_error`, which tests exactly this property.


In [ ]:
def frame_codec() -> type[RPCFrameCodec]:
    return RPCFrameCodec

_make_frame_codec = frame_codec

frame_codec = _make_frame_codec()

bad_header = b"\xFF" + b"\x00" * (HEADER_SIZE - 1)
with pytest.raises(ValueError, match="Invalid protocol magic byte"):
    frame_codec.decode_header(bad_header)

print('PASSED: test_invalid_magic_byte_raises_error')

## 4. Measure it: Rpc successful method invocation

An assertion tells you a property holds. A measurement tells you *how much*.
This cell runs `test_rpc_successful_method_invocation` and times it.


In [ ]:
import time

_t0 = time.perf_counter()

async def rpc_system() -> tuple[RPCServer, RPCClient]:
    server = RPCServer()

    # Register demo procedures
    async def add(a: int, b: int) -> int:
        return a + b

    async def slow_fetch(item_id: str, delay_sec: float = 0.05) -> dict:
        await asyncio.sleep(delay_sec)
        return {"id": item_id, "status": "ACTIVE"}

    async def fail_op() -> None:
        raise ValueError("Database unavailable")

    server.register("add", add)
    server.register("slow_fetch", slow_fetch)
    server.register("fail_op", fail_op)

    client = RPCClient(server)
    return server, client

_make_rpc_system = rpc_system

rpc_system = await _make_rpc_system()

_, client = rpc_system
result = await client.invoke("add", {"a": 15, "b": 27})
assert result == 42

_elapsed = (time.perf_counter() - _t0) * 1000
print('PASSED: test_rpc_successful_method_invocation')
print(f'wall clock: {_elapsed:.2f} ms')

## 5. 🛠️ Fix this cell — it is deliberately broken

The cell below asserts something **false** about the real object. Read the
failure, work out the true value from the module's actual behaviour, and correct
the expected number.

Do not delete the assertion. The point is to make it pass by knowing the answer.


In [ ]:
# DELIBERATELY BROKEN - fix the expected value below.
# Hint: print the real value first, then decide what the assertion should say.

exports = [n for n in dir(rpc_protocol_engine) if not n.startswith('_')]
print(f'actual export count: {len(exports)}')
print(f'actual exports     : {exports}')

EXPECTED_EXPORT_COUNT = 999      # <-- wrong on purpose. Replace it.

assert len(exports) == EXPECTED_EXPORT_COUNT, (
    f'expected {EXPECTED_EXPORT_COUNT} exports, found {len(exports)}. '
    'Read the printed value above and correct the constant.'
)
print('Fixed - assertion now reflects reality.')

### 🎓 Key takeaways

1. Protocol choice is a latency and bandwidth decision, not a style preference.
2. Binary framing plus multiplexing removes head-of-line blocking at the HTTP layer.
3. REST for public APIs, gRPC for internal service-to-service - and know why.

---

**Continue with this module:**

- [README.md](README.md) — the mental model and failure modes
- [PROJECT_GUIDE.md](PROJECT_GUIDE.md) — build it yourself, in 3 tiers
- [starter/](starter/) — your stubs; run the tests from there to grade yourself
- [debug_lab/SYMPTOMS.md](debug_lab/SYMPTOMS.md) — diagnose planted bugs from the symptom
- [TROUBLESHOOTING_AND_EDGE_CASES.md](TROUBLESHOOTING_AND_EDGE_CASES.md) — real errors, real causes
- [SELF_ASSESSMENT_AND_CHALLENGES.md](SELF_ASSESSMENT_AND_CHALLENGES.md) — quiz and diagnostics
